# L2c: Arrays, Dictionaries, Tables, and Collection Operations

Representation should follow the operations a problem requires. We will move one small set of warehouse shift records through vectors, dictionaries, named tuples, and a `DataFrame`, and watch which questions each one makes easy.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Choose a collection from the operations you need:__ Select among arrays, dictionaries, named tuples, and tables based on how the data will be accessed and transformed, rather than reaching for whichever container is most familiar.
> * __Distinguish mutating from non-mutating operations:__ Recognize which operations modify a collection in place and which return a new one, and understand why Julia marks the difference with a trailing exclamation mark.
> * __Ask questions of tabular data:__ Filter, transform, group, and summarize records so that the code expresses the question being asked rather than the mechanics of moving values around.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

Besides Julia's `Base` library, this lecture uses [the `DataFrames.jl` package](https://dataframes.juliadata.org/stable/) for the table work and [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) for the checks at the end. `Include.jl` loads both.

___

## The data we will carry through

Every representation in this lecture holds the same measurements, so that the differences between them are differences of _structure_ rather than of content.

> __The setting:__
>
> A regional fulfillment center runs two picking zones, `East` and `West`. For each of six shifts we recorded the number of customer orders completed and the labor hours worked. Dividing one by the other gives __throughput__ in orders per labor hour, which is the number the operations manager actually cares about.

This is the setting the course returns to: a distribution network with zones, shifts, orders, and capacity. Week 3 meets these same records again as a file that arrived from somewhere else and has to be checked before it can be trusted.

___

___

## Arrays: ordered numerical data

Start with the simplest case: one number per shift, in shift order. A [`Vector`](https://docs.julialang.org/en/v1/base/arrays/#Base.Vector) stores those six order counts contiguously, which makes two things cheap — reading the $i$-th element, and doing arithmetic on all of them at once.

> __What a vector is good at:__
>
> * __Position is meaning.__ `orders_completed[1]` is shift 1. The index carries information, which is exactly why a vector is wrong for data that has no natural order.
> * __Elementwise arithmetic is one operation.__ Writing `orders_completed ./ 8.0` divides every element and returns a new vector. That dot is [broadcasting](https://docs.julialang.org/en/v1/base/arrays/#Base.Broadcast.broadcast), and it is how Julia expresses "do this to each element" without a loop.
> * __Selecting several at once.__ Indexing with a vector of indices, `orders_completed[[2, 5]]`, returns just those shifts, again as a new vector.

Note what broadcasting does _not_ do: it leaves `orders_completed` unchanged and hands back a new vector. Keep an eye on that distinction — it comes back at the end of this notebook.

Let's build the vector and take all three of those operations for a spin:

In [ ]:
orders_completed = [92, 102, 108, 87, 100, 96]
orders_per_eight_hour_shift = orders_completed ./ 8.0
(first_shift = orders_completed[1],
 selected_shifts = orders_completed[[2, 5]],
 per_eight_hours = orders_per_eight_hour_shift)

Three results, three access patterns: one element by position, a subset by a list of positions, and every element transformed at once. None of them changed `orders_completed` itself.

That third result is already slightly dishonest, though. It assumed every shift was eight hours long, and they were not. Holding the hours somewhere is the next problem.

___

___

## Dictionaries and named records

A vector fails as soon as the thing you want to look up is not a position. What are the units of the throughput column? There is no "third element" answer to that question — the natural key is a name.

> __Two different jobs, two different containers:__
>
> * __A [`Dict`](https://docs.julialang.org/en/v1/base/collections/#Base.Dict) maps keys to values.__ Here the keys are column names and the values are unit strings. Lookup is by key rather than position, at the cost of hashing on every access and storing the keys alongside the values. Dictionaries keep no order — do not rely on the printed sequence.
> * __A [named tuple](https://docs.julialang.org/en/v1/base/base/#Core.NamedTuple) is one record.__ It is immutable and its field names are part of its type, so `records[1].zone` is checked when the code is compiled rather than looked up when it runs. That makes it the right shape for a single shift, and the wrong shape for a collection you need to grow.

The pairing below is the common one: a dictionary for metadata _about_ the columns, and a vector of named tuples for the shifts themselves. Now each shift carries its own labor hours, so the eight-hour assumption is gone.

Let's build both:

In [ ]:
units = Dict(
    :orders_completed => "orders",
    :labor_hours => "h",
    :orders_per_labor_hour => "orders/h",
)
records = [
    (shift = 1, zone = "East", orders_completed = 92,  labor_hours = 8.0),
    (shift = 2, zone = "East", orders_completed = 102, labor_hours = 8.0),
    (shift = 3, zone = "East", orders_completed = 108, labor_hours = 9.0),
    (shift = 4, zone = "West", orders_completed = 87,  labor_hours = 7.5),
    (shift = 5, zone = "West", orders_completed = 100, labor_hours = 8.0),
    (shift = 6, zone = "West", orders_completed = 96,  labor_hours = 7.5),
]
(throughput_unit = units[:orders_per_labor_hour], first_record = records[1])

We can now ask both kinds of question: what does this column mean, and what happened on shift 1. Notice that `records` is a plain vector of six named tuples — the schema is real, but nothing enforces it yet, and nothing lets us operate on a whole column at once.

That is the gap a table closes.

___

___

## Tables: columns plus row relationships

A [`DataFrame`](https://dataframes.juliadata.org/stable/) is what a vector of records becomes when you want to work with the columns as well as the rows. It stores each column contiguously, like six little vectors that agree on their length, while still letting you talk about row 4 as a unit.

> __Why this beats a vector of named tuples:__
>
> Adding a derived quantity to a vector of records means rebuilding all six records. In a table it is one assignment to a new column, computed by broadcasting across the columns you already have. Below we add `orders_per_labor_hour`, the throughput, as the order count divided by the labor hours.

The `shifts::DataFrame` variable below holds one row per shift, with a derived `orders_per_labor_hour` column:

In [ ]:
shifts = let
    table = DataFrame(records) # one row per record, columns from the record fields
    table.orders_per_labor_hour = table.orders_completed ./ table.labor_hours # derived
    table # return the populated table
end

The operations target is __12 orders per labor hour__. Which shifts met it?

[The `filter(...)` function](https://dataframes.juliadata.org/stable/lib/functions/#Base.filter) returns the selected rows as a new table, leaving `shifts` untouched. The predicate states the acceptance rule directly, so the code reads as the question being asked.

In [ ]:
shifts_meeting_target = filter(row -> row.orders_per_labor_hour >= 12.0, shifts)
shifts_meeting_target

Four of the six shifts clear the target. `shifts` still has all six — the filter handed back a new table rather than editing the old one.

___

___

## Group and summarize

Filtering answers questions about individual rows. "Which zone is performing better?" is a question about _sets_ of rows, and it needs a different move: split the table into groups, compute something for each group, then stitch the answers back into one table.

> __Split, apply, combine:__
>
> * [The `groupby(...)` function](https://dataframes.juliadata.org/stable/lib/functions/#DataAPI.groupby) splits the table on the values in a column. Grouping on `:zone` gives two groups of three shifts. Nothing is aggregated yet, and no row is lost — the rows are simply partitioned.
> * [The `combine(...)` function](https://dataframes.juliadata.org/stable/lib/functions/#DataFrames.combine) applies a reduction to each group and returns one row per group. The `:column => function => :new_name` syntax names both what goes in and what comes out, so the resulting table documents itself.

We ask for three things per zone: the mean throughput using [the `mean(...)` function](https://docs.julialang.org/en/v1/stdlib/Statistics/#Statistics.mean), the total orders using `sum`, and the number of shifts using [the `nrow` helper](https://dataframes.juliadata.org/stable/lib/functions/#DataAPI.nrow). Then we sort so the output order is deterministic rather than dependent on which group happened to be built first.

So which zone wins?

In [ ]:
zone_summary = combine(
    groupby(shifts, :zone),
    :orders_per_labor_hour => mean => :mean_throughput,
    :orders_completed => sum => :total_orders,
    nrow => :number_of_shifts,
)
sort!(zone_summary, :zone)

Read those two columns against each other before answering.

__East completed more orders__ — $302$ against $283$. __West was more productive per hour__ — $12.30$ against $12.08$. Neither zone is simply "better". East did more work; West did it faster. Which number matters depends on whether you are trying to clear a backlog or control labor cost, and the table cannot make that choice for you.

> __One more mutation to notice:__ [the `sort!(...)` function](https://docs.julialang.org/en/v1/base/sort/#Base.sort!) ends in an exclamation mark. That is Julia's convention for a function that modifies its argument in place, and it is the counterpart to everything else in this notebook, which has quietly been returning new objects and leaving the originals alone. When you read unfamiliar Julia, the `!` tells you whether your data is about to change underneath you.

___

Finally, a summary of the throughput column on its own, using a function we wrote rather than one the package supplies.

[The `measurement_summary(...)` function](src/Compute.jl) takes any numerical vector and returns a named tuple holding the count, minimum, maximum, and mean. It rejects an empty vector and rejects one containing non-finite values, because a summary of nothing, or a mean poisoned by a `NaN`, is worse than an error. It also leaves its argument untouched.

We will reuse it in the labs, which is the whole reason it lives in [`src/Compute.jl`](src/Compute.jl) rather than in a cell here:

In [ ]:
throughput_report = measurement_summary(shifts.orders_per_labor_hour)

Six shifts, ranging from $11.50$ to $12.80$ orders per labor hour, averaging about $12.19$. That column did not exist when the notebook started — it was derived from two others — and it has now been through four representations and come back out as a summary.

___

___

Each test below pins one claim made above: that the vector, dictionary, and table hold what we said they hold, and that the filtering and grouping produced the counts we expected.

Do they all pass?

In [ ]:

@testset "collections and tables" begin
    @test orders_completed[3] == 108
    @test units[:labor_hours] == "h"
    @test nrow(shifts) == 6
    @test nrow(shifts_meeting_target) == 4
    @test names(shifts) == ["shift", "zone", "orders_completed", "labor_hours", "orders_per_labor_hour"]
    @test zone_summary.number_of_shifts == [3, 3]
    @test zone_summary.total_orders == [302, 283]
    @test zone_summary.mean_throughput[2] > zone_summary.mean_throughput[1]
    @test throughput_report.count == 6
    @test throughput_report.minimum == 11.5
    @test throughput_report.maximum == 12.8
end

___

## Summary
The same shift records can live in a vector, a dictionary, a set of named tuples, or a table, and the right choice is dictated by the operations the problem actually requires.

> __Key Takeaways:__
>
> * **Access pattern picks the container:** Arrays give ordered, positional access and cheap elementwise arithmetic, while dictionaries give lookup by key at the cost of hashing and storage, so the operation you need most often should decide.
> * **Records and tables scale differently:** A named tuple describes one record clearly, but many records sharing one schema belong in a table where columns can be selected, filtered, grouped, and derived from one another as a unit.
> * **Transformations should read as questions:** Filtering, grouping, and aggregation are most useful when the code states the question being asked of the data rather than the mechanics of rearranging it, which is also what makes the answer arguable.

Week 3 meets these same shift records again, this time as a file that arrived from somewhere else, where the schema is something you discover and defend rather than something you declared.
___